# Kernel fusion, launch overhead and CUDA graphs on a T4

Companion to the card **fusion-cuda-graphs.html** (perf-3-kernels). Everything here is PyTorch; no CUDA C++.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

What you will measure:

1. **Fusion.** `x.cos().cos()` and a 10-op elementwise chain on 64M floats (256 MiB): eager PyTorch (one kernel per op) vs `torch.compile` (one fused Triton kernel). Printed as ms and GB/s next to the T4's 320 GB/s.
2. **Horace He's repeat-multiply sweep.** `x = x * c` repeated 1 … 512 times inside one fused kernel. The runtime stays flat while the kernel is memory-bound, then climbs once it becomes compute-bound.
3. **Launch overhead.** A chain of 20 tiny kernels run eagerly vs captured once with `torch.cuda.CUDAGraph` and replayed. Printed as µs per step. Plus Horace's overhead test: double the size and see whether the time doubles.
4. **(Optional)** a `torch.profiler` trace that shows the CPU launches and the GPU kernels.

Timing uses `torch.cuda.Event` after warm-up. Compile time is excluded: every compiled function runs a few times before the clock starts.
Colab's T4 clocks and availability vary, so expect some spread between runs.

In [ ]:
!nvidia-smi

In [ ]:
import time, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
dev = "cuda"
print(torch.__version__, torch.cuda.get_device_name(0))

PEAK_BW = 320e9        # T4 peak DRAM bandwidth, Turing whitepaper (datasheet prints 300 GB/s)
PEAK_FP32 = 8.1e12     # T4 FP32, FMA counted as 2 FLOPs

def time_ms(fn, iters=20, warmup=5):
    """Average GPU time per call in ms, measured with CUDA events after warm-up."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / iters

def fresh_compile(f):
    """Compile f with a clean Dynamo cache, so each variant gets its own fused kernel."""
    torch._dynamo.reset()
    return torch.compile(f)

## 1. Fusion: eager vs `torch.compile`

Eager PyTorch launches one kernel per op, and every kernel reads its input from DRAM and writes its output back.
`x.cos().cos()` therefore moves 4 × 256 MiB; the fused version reads `x` once and writes the result once: 2 × 256 MiB (Horace He's example).
The 10-op chain (5 × multiply, 5 × add) moves 20 × 256 MiB eagerly and still 2 × 256 MiB fused.

GB/s below = bytes that version actually has to move ÷ time. Both should sit near 320 GB/s if the kernels are memory-bound; the fused one wins by moving fewer bytes, not by running faster per byte.

In [ ]:
N = 64 * 2**20                      # 64M floats = 256 MiB
x = torch.randn(N, device=dev)
nbytes = x.numel() * x.element_size()

def coscos(x):
    return x.cos().cos()

def chain10(x):
    for _ in range(5):
        x = x * 1.01
        x = x + 0.1
    return x

rows = []
for name, f, eager_accesses in [("x.cos().cos()", coscos, 4), ("10-op chain", chain10, 20)]:
    fc = fresh_compile(f)
    ref = f(x)
    out = fc(x)                     # first call compiles; not timed
    torch.testing.assert_close(out, ref, rtol=1e-4, atol=1e-4)
    t_eager = time_ms(lambda: f(x))
    t_comp = time_ms(lambda: fc(x))
    bw_e = eager_accesses * nbytes / (t_eager * 1e-3)
    bw_c = 2 * nbytes / (t_comp * 1e-3)
    rows.append((name, t_eager, bw_e, t_comp, bw_c))
    print(f"{name:14s} eager {t_eager:7.2f} ms ({eager_accesses} x 256 MiB, {bw_e/1e9:5.0f} GB/s = {bw_e/PEAK_BW:4.0%} of 320) | "
          f"compiled {t_comp:6.2f} ms (2 x 256 MiB, {bw_c/1e9:5.0f} GB/s = {bw_c/PEAK_BW:4.0%}) | speedup {t_eager/t_comp:4.1f}x")

print("\nFloor at 320 GB/s: 2 x 256 MiB =", round(2 * nbytes / PEAK_BW * 1e3, 2), "ms;",
      "4 x 256 MiB =", round(4 * nbytes / PEAK_BW * 1e3, 2), "ms; 20 x 256 MiB =", round(20 * nbytes / PEAK_BW * 1e3, 2), "ms")

## 2. Horace He's repeat-multiply sweep

One fused kernel that reads `x`, multiplies it by a constant `repeat` times, and writes it back. The memory traffic is fixed (2 × 4 bytes per element); only the compute grows.
Horace measured this on an A100: runtime "doesn't increase noticeably at all until we're performing 64 multiplications", achieved FLOPS start at ~0.2 TFLOPS and rise to near 9.75 TFLOPS (the A100's non-FMA FP32 rate).

Our prediction for the T4 (not measured): 320 GB/s ÷ 8 bytes = 40G elements/s, and 8.1 TFLOPS counts an FMA as 2, so ~4.05T plain multiplies/s. The knee should sit near 4.05e12 ÷ 40e9 ≈ **100 multiplies per element**.

In [ ]:
N2 = 16 * 2**20                     # 16M floats = 64 MiB, keeps the sweep quick
x2 = torch.randn(N2, device=dev)

def make_f(repeat):
    def f(x):
        for _ in range(repeat):
            x = x * 1.0001
        return x
    return f

print(f"{'repeat':>6} {'ms':>8} {'GB/s':>7} {'% of 320':>8} {'GFLOP/s':>8}")
for repeat in [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]:
    fc = fresh_compile(make_f(repeat))
    fc(x2)                          # compile, not timed
    t = time_ms(lambda: fc(x2), iters=20)
    bw = 2 * 4 * N2 / (t * 1e-3)
    flops = N2 * repeat / (t * 1e-3)
    print(f"{repeat:6d} {t:8.3f} {bw/1e9:7.0f} {bw/PEAK_BW:8.0%} {flops/1e9:8.0f}")

## 3. Launch overhead: eager vs CUDA graph replay

A "step" is 20 dependent multiplies, each its own kernel (like the NVIDIA CUDA Graphs blog's 20 short kernels).
With a tiny tensor each kernel's actual work is well under a microsecond, so the step time is almost all overhead: Python, the PyTorch dispatcher, and the kernel launch.

A CUDA graph records the 20 launches once (capture) and replays them with one call. The kernels are identical; only the way they are submitted changes.
Capture follows the PyTorch recipe: warm up on a side stream, capture into a `torch.cuda.CUDAGraph`, then copy new inputs into the captured (static) input tensor and call `replay()`.

In [ ]:
def step20(x):
    for _ in range(20):
        x = x * 1.0001
    return x

def graph_for(fn, static_in):
    s = torch.cuda.Stream()
    s.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(s):
        for _ in range(3):
            fn(static_in)
    torch.cuda.current_stream().wait_stream(s)
    g = torch.cuda.CUDAGraph()
    with torch.cuda.graph(g):
        static_out = fn(static_in)
    return g, static_out

STEPS = 200
print(f"{'elements':>10} {'eager us/step':>14} {'graph us/step':>14} {'speedup':>8}")
for n in [1024, 500_000]:            # 500,000 = the NVIDIA blog's array size
    static_in = torch.randn(n, device=dev)
    t0 = time.perf_counter()
    g, static_out = graph_for(step20, static_in)
    torch.cuda.synchronize()
    capture_ms = (time.perf_counter() - t0) * 1e3
    ref = step20(static_in)
    g.replay(); torch.cuda.synchronize()
    torch.testing.assert_close(static_out, ref)
    t_e = time_ms(lambda: step20(static_in), iters=STEPS, warmup=20) * 1e3
    t_g = time_ms(g.replay, iters=STEPS, warmup=20) * 1e3
    print(f"{n:10,d} {t_e:14.1f} {t_g:14.1f} {t_e/t_g:7.1f}x   (capture incl. warm-up: {capture_ms:.1f} ms, once)")
print("per kernel = the numbers above / 20")

### Horace's overhead test: double the size, does the time double?

"If you double your batch size but your runtime only increases by 10%, you're likely overhead bound." Run the eager 20-kernel step at n and 2n for a tiny and a large tensor.

In [ ]:
for n in [1024, 32 * 2**20]:
    a = torch.randn(n, device=dev); b = torch.randn(2 * n, device=dev)
    t1 = time_ms(lambda: step20(a), iters=50, warmup=10)
    t2 = time_ms(lambda: step20(b), iters=50, warmup=10)
    verdict = "overhead-bound" if t2 / t1 < 1.3 else "work-bound (memory here)"
    print(f"n = {n:>11,d}: {t1*1e3:9.1f} us -> 2n: {t2*1e3:9.1f} us   ratio {t2/t1:4.2f}  -> {verdict}")

## 4. (Optional) See the launches in a profiler trace

`torch.profiler` records CPU-side ops (including `cudaLaunchKernel`) and GPU kernels. Download `trace.json` from the file browser on the left and open it in Perfetto or `chrome://tracing`. In the eager part, look for the gaps between the tiny GPU kernels; in the graph part, one `cudaGraphLaunch` on the CPU row and the kernels back to back on the GPU row.

In [ ]:
from torch.profiler import profile, ProfilerActivity
small = torch.randn(1024, device=dev)
g_small, _ = graph_for(step20, small)
torch.cuda.synchronize()
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    for _ in range(5):
        step20(small)
    torch.cuda.synchronize()
    for _ in range(5):
        g_small.replay()
    torch.cuda.synchronize()
print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=12))
prof.export_chrome_trace("trace.json")
print("wrote trace.json")

## Reference numbers from the sources (not T4)

**Horace He, "Making Deep Learning Go Brrrr From First Principles" (2022), A100:** 1.5 TB/s and 19.5 TFLOPS non-matmul, so "until you're doing about a hundred operations in your unary operator, you'll be spending more time performing memory accesses than actual compute". Repeat-multiply sweep: flat runtime up to 64 multiplies, ~0.2 → ~9.75 TFLOPS. `x.cos().cos()`: 4 global reads+writes eager, 2 fused. PyTorch on tiny tensors: ~280 thousand ops/s.

**Alan Gray, NVIDIA, "Getting Started with CUDA Graphs" (2019), Tesla V100, 20 kernels × 1,000 steps, 500,000 elements:**

| Launch style | Time per kernel |
|---|---|
| kernel execution alone | 2.9 µs |
| launch + sync after every kernel | 9.6 µs |
| sync once per step (launches overlap) | 3.8 µs |
| one CUDA graph per step | 3.4 µs |

Graph creation + instantiation ≈ 400 µs, once.

**Hazy Research, "Look Ma, No Bubbles!" (2025), H100:** launch cost ~2.1 µs on a stream, ~1.3 µs with CUDA graphs (dummy kernel).

Your Colab numbers will differ: the T4 is slower than these GPUs, eager PyTorch adds Python and dispatcher time that the C++ blog does not have, and Colab's CPU is shared.

## Try this

- `torch.compile(step20, mode="reduce-overhead")`: this mode uses CUDA graphs under the hood. Does it match the manual graph, or beat it by also fusing the 20 multiplies into one kernel?
- In part 1, switch to `torch.float16`. Bytes halve, so the times should roughly halve; the eager/compiled ratio should not change.
- In part 3, change the step to 100 kernels, and try `n = 8 * 2**20`. At what size does the graph stop helping?